### Methods for Structured Output in LangChain

LangChain provides several methods to extract structured data, ranging from basic text parsing to advanced native model features. 

#### 1. `.with_structured_output()` (Best Practice)
The modern, recommended approach. It abstracts away the complexity by automatically using the best underlying native feature (like function calling or JSON mode) supported by the chosen LLM. 
* **Input:** Pydantic, TypedDict, or Dataclasses.
* **Output:** A strongly formatted object matching your schema.

#### 2. Output Parsers (Formatting & Parsing)
Used when a model doesn't support native tool calling, or when you need a very specific format. Parsers inject formatting instructions into the prompt and parse the string response.
* **`PydanticOutputParser`**: Extracts data into a Pydantic object.
* **`JsonOutputParser`**: Parses the output into a standard JSON dictionary.
* **`CommaSeparatedListOutputParser`**: Returns a list of items parsed from a comma-separated text string.
* **`DatetimeOutputParser`**: Parses LLM text into a Python `datetime` object.
* **`XMLOutputParser`**: Parses output formatted as XML into a dictionary.
* **`StructuredOutputParser`**: A legacy parser for defining schemas without Pydantic.

#### 3. Native JSON Mode
Many providers (like OpenAI, Google Gemini, Anthropic) support a native "JSON Mode". When enabled (e.g., passing `{ "response_format": { "type": "json_object" } }` in OpenAI), the model is guaranteed to output valid JSON. You must still instruct the model to output JSON in your system prompt.

#### 4. Tool/Function Calling (`.bind_tools()`)
Before `.with_structured_output()` existed, the standard way to get structured data was by binding a dummy "tool" or "function" to the model and forcing the model to call it. The LLM returns a `ToolCall` containing the structured arguments as a dictionary.

#### 5. Output Fixing & Retry Parsers 
Often, LLMs make small syntax mistakes when generating structured output (like a missing JSON bracket). LangChain offers parsers to automatically handle exceptions:
* **`OutputFixingParser`**: Wraps another parser. If the first parser fails, it passes the misformatted output to a newly invoked LLM to try and fix the formatting.
* **`RetryOutputParser`**: Similar to the fixing parser, but it passes both the original prompt and the misformatted output to the LLM to get a better retry without losing context.
```

### Pydantic is the richest model set which provides field validation , descriptions and nested structures.

In [9]:
from langchain.chat_models import init_chat_model
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000216B9CF5E50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000216B9CF6850>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [17]:
# Schema about movie descriptions using pydantic
from pydantic import BaseModel ,Field
class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year teh movie was released")
    director:str=Field(description="The director of the movie")
    rating:str=Field(description="The movies rating out of 10")


In [13]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000216B9CF5E50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000216B9CF6850>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year teh movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'string'}}, 'required': ['title', 'year', 'd

In [14]:
response1=model.invoke("Provide details about the movie Inception")
print(response1.content)


<think>
Okay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. It's a 2010 film directed by Christopher Nolan, right? The title is "Inception," which I think means planting an idea into someone's subconscious. The main character is Dom Cobb, played by Leonardo DiCaprio. He's a thief who steals information by entering people's dreams. Instead of stealing, he's now trying to plant an idea, which is the inception part.

The cast includes some big names. There's Joseph Gordon-Levitt, who does a lot of physical stunts, and Ellen Page plays Ariadne, a new architect in the dream-sharing world. Tom Hardy is also in it, as the antagonist, maybe? I think the plot involves multiple layers of dreams, each deeper than the last, and the risk of getting trapped in the dream world is a big part of the story. There's a concept called limbo, where people who get stuck in their own subconscious go.

The special effects must be really advanced, like t

In [ ]:
response2=model_with_structure.invoke("Provide details about the movie Inception")
print(response2)

title='Inception' year=2010 director='Christopher Nolan' rating='8.8'


### Messaged output along with parsed structure

In [24]:
# Schema about movie descriptions using pydantic
from pydantic import BaseModel ,Field
class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year teh movie was released")
    director:str=Field(description="The director of the movie")
    rating:str=Field(description="The movies rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response2=model_with_structure.invoke("Provide details about the movie Inception")
print(response2)


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me see what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. I need to provide those parameters. \n\nFirst, I know that Inception is directed by Christopher Nolan. The title is Inception. The release year was 2010. The rating... maybe I should check a reliable source, but since I'm making this up, I'll say it's 8.8/10, which is a common high rating for that movie. Let me make sure all required fields are included. Title, year, director, rating—yes, all there. So I'll structure the tool call with those details.\n", 'tool_calls': [{'id': 'z009jky0v', 'function': {'arguments': '{"director":"Christopher Nolan","rating":"8.8","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 203, 'prompt_tokens': 230, 'total_tokens': 

### Nested structure

In [28]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    Budget:float | None =Field(None,description="Budget in million Usd")

model_with_structure=model.with_structured_output(MovieDetails)

response2=model_with_structure.invoke("Provide details about the movie Inception")
response2

MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Jack')], genres=['Science Fiction', 'Action', 'Thriller'], Budget=160.0)

### TypedDict
TypedDict provides a simpler alternative using python's builtin typing ideal when you dont need runtime validation

In [32]:
from typing_extensions import TypedDict,Annotated
class Movie(TypedDict):
    """A movie with details."""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year teh movie was released")
    director:str=Field(description="The director of the movie")
    rating:str=Field(description="The movies rating out of 10")

model_with_typeddict=model.with_structured_output(Movie)

response2=model_with_typeddict.invoke("Provide details about the movie Inception")
response2

{'director': 'Christopher Nolan',
 'rating': '8.8',
 'title': 'Inception',
 'year': 2010}

In [34]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    Budget:float | None =Field(None,description="Budget in million USD")

model_with_structure=model.with_structured_output(MovieDetails)

response2=model_with_structure.invoke("Provide details about the movie Avengers")
response2


{'Budget': 245000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Superhero'],
 'title': 'Avengers',
 'year': '2012'}

In [35]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses
Dataclass is a classical typically containing mainly data although there arrent any restrictions . You can  create it using @dataclass decorator

In [39]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [44]:
from pydantic import BaseModel,Field
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email of the person")
    phone:str=Field(description="The phone number of the person")

agent=create_agent(
    model = init_chat_model(
    "gpt-5",
    model_provider="openai", 
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"]
    ),
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact from Arnab Mandal , arnabmandal261@gmail.com ,+91 9830945015"}]
})
result["structured_response"]

ContactInfo(name='Arnab Mandal', email='arnabmandal261@gmail.com', phone='+91 9830945015')

In [45]:
#Dataclass
from dataclasses import dataclass
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of a person"""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email of the person")
    phone:str=Field(description="The phone number of the person")

agent=create_agent(
    model = init_chat_model(
    "gpt-5",
    model_provider="openai", 
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENAI_API_KEY"]
    ),
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact from Arnab Mandal , arnabmandal261@gmail.com ,+91 9830945015"}]
})
result["structured_response"]

ContactInfo(name='Arnab Mandal', email='arnabmandal261@gmail.com', phone='+91 9830945015')